In [16]:
import json
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 1. LOAD DỮ LIỆU
# ==========================================
print(f"--- Đang chạy kiểm thử với THRESHOLD = 0.78 ---")

with open('./data/val.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
qid_to_text = {str(item['qid']): item['question'] for item in raw_data}

with open('val_embeddings.json', 'r', encoding='utf-8') as f:
    embedding_list = json.load(f)
qid_to_emb = {str(item['qid']): item['embedding'] for item in embedding_list}

with open('val_labels.json', 'r', encoding='utf-8') as f:
    label_data = json.load(f)

# ==========================================
# 2. CHUẨN BỊ INPUT
# ==========================================
final_true_labels = []
final_pred_labels = []
final_qids = []

X_model_list = []
y_model_list = []
qids_model_list = [] 

# Duyệt dữ liệu để tách Rule 1
for qid, label in label_data.items():
    qid = str(qid)
    label = int(label)
    
    if qid not in qid_to_emb or qid not in qid_to_text: continue

    content = qid_to_text[qid]
    
    # --- RULE 1: Class 0 ---
    if content.strip().startswith("Đoạn thông tin"):
        final_true_labels.append(label)
        final_pred_labels.append(0)
        final_qids.append(qid)
        continue 
    
    # Gom lại để chạy Model
    X_model_list.append(qid_to_emb[qid])
    y_model_list.append(label)
    qids_model_list.append(qid)

# ==========================================
# 3. CHẠY MODEL VÀ ÁP DỤNG THRESHOLD 0.78
# ==========================================
X_model = np.array(X_model_list)
y_model = np.array(y_model_list)

if len(X_model) > 0:
    # Model Config
    svm_clf = SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42)
    knn_clf = KNeighborsClassifier(n_neighbors=9) 
    ensemble_model = VotingClassifier(
        estimators=[('svm', svm_clf), ('knn', knn_clf)],
        voting='soft',
        weights=[1, 1] 
    )

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Lấy xác suất
    y_probs = cross_val_predict(ensemble_model, X_model, y_model, cv=kf, method='predict_proba')
    
    # Xử lý Logic Threshold
    THRESHOLD = 0.78
    
    for i, probs in enumerate(y_probs):
        pred_label = np.argmax(probs) + 1 # Nhãn gốc từ Model (1 hoặc 2)
        confidence = np.max(probs)        # Độ tự tin
        
        current_qid = qids_model_list[i]
        content = qid_to_text[current_qid]
        digit_count = sum(c.isdigit() for c in content)
        
        # --- LOGIC QUYẾT ĐỊNH ---
        final_decision = pred_label # Mặc định tin model
        
        # Nếu Model không đủ tự tin (Dưới 78%) VÀ câu ít số
        if confidence < THRESHOLD and digit_count < 2:
            final_decision = 2
            
        final_true_labels.append(y_model[i])
        final_pred_labels.append(final_decision)
        final_qids.append(current_qid)

# ==========================================
# 4. KẾT QUẢ
# ==========================================
print("\n=== KẾT QUẢ CUỐI CÙNG ===")
y_true = np.array(final_true_labels)
y_pred = np.array(final_pred_labels)

target_names = ['Lớp 0 (Đoạn thông tin)', 'Lớp 1 (STEM)', 'Lớp 2 (General)']
print(classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=target_names, zero_division=0))
print(f"Total Accuracy: {accuracy_score(y_true, y_pred):.4f}")

# Soi lại các câu sai
if accuracy_score(y_true, y_pred) < 1.0:
    print("\n[Các câu vẫn còn SAI]")
    for i in range(len(y_true)):
        if y_true[i] != y_pred[i]:
            print(f"❌ QID: {final_qids[i]} | True: {y_true[i]} -> Pred: {y_pred[i]}")
            # In ra thông tin để debug xem tại sao threshold không bắt được (hoặc bắt nhầm)
            content = qid_to_text[final_qids[i]]
            print(f"   Text: {content[:50]}...")
else:
    print("\n>>> TUYỆT VỜI! THRESHOLD 0.78 ĐÃ GIẢI QUYẾT HẾT VẤN ĐỀ.")

--- Đang chạy kiểm thử với THRESHOLD = 0.78 ---

=== KẾT QUẢ CUỐI CÙNG ===
                        precision    recall  f1-score   support

Lớp 0 (Đoạn thông tin)       1.00      1.00      1.00        20
          Lớp 1 (STEM)       1.00      0.94      0.97        34
       Lớp 2 (General)       0.95      1.00      0.97        39

              accuracy                           0.98        93
             macro avg       0.98      0.98      0.98        93
          weighted avg       0.98      0.98      0.98        93

Total Accuracy: 0.9785

[Các câu vẫn còn SAI]
❌ QID: val_0013 | True: 1 -> Pred: 2
   Text: Khẳng định nào là đúng về chú thích trong Python?...
❌ QID: val_0031 | True: 1 -> Pred: 2
   Text: Khí nhà kính nào chiếm khoảng một nửa khối lượng k...


In [ ]:
import json
import numpy as np
import joblib # Thư viện để lưu model
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier

# 1. Load dữ liệu
print("--- Đang tải dữ liệu để huấn luyện ---")
with open('./data/val.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
qid_to_text = {str(item['qid']): item['question'] for item in raw_data}

with open('val_embeddings.json', 'r', encoding='utf-8') as f:
    embedding_list = json.load(f)
qid_to_emb = {str(item['qid']): item['embedding'] for item in embedding_list}

with open('val_labels.json', 'r', encoding='utf-8') as f:
    label_data = json.load(f)

# 2. Lọc dữ liệu sạch để Train Model (Chỉ lấy Class 1 và 2)
# Lưu ý: Class 0 (Rule 1) không cần đưa vào train vì ta xử lý bằng code if-else
X_train_list = []
y_train_list = []

for qid, label in label_data.items():
    qid = str(qid)
    label = int(label)
    
    # Bỏ qua nếu thiếu dữ liệu
    if qid not in qid_to_emb or qid not in qid_to_text: continue
    
    # Bỏ qua nếu là nhãn 0 (để Rule lo) hoặc nhãn rác
    if label not in [1, 2]: continue
    
    # Bỏ qua các câu "Đoạn thông tin..." vì Rule 1 sẽ chặn, không cần model học làm gì cho nhiễu
    content = qid_to_text[qid]
    if content.strip().startswith("Đoạn thông tin"): continue

    X_train_list.append(qid_to_emb[qid])
    y_train_list.append(label)

X_train = np.array(X_train_list)
y_train = np.array(y_train_list)

print(f"-> Số lượng mẫu sạch dùng để train Model: {len(X_train)}")

# 3. Khởi tạo và Train Model
print("--- Đang huấn luyện mô hình (Full Data) ---")
svm_clf = SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42)
knn_clf = KNeighborsClassifier(n_neighbors=9)
ensemble_model = VotingClassifier(
    estimators=[('svm', svm_clf), ('knn', knn_clf)],
    voting='soft',
    weights=[1, 1]
)

# Train trên toàn bộ tập dữ liệu (Fit)
ensemble_model.fit(X_train, y_train)

# 4. Lưu Model
model_filename = 'ensemble_model.pkl'
joblib.dump(ensemble_model, model_filename)
print(f"✅ Đã lưu mô hình thành công vào file: {model_filename}")